# **Pre procesamiento de datos censales**

Por: Daning Montaño - Ocampo

Se unen diferentes capas provenientes de los censos para poder espacializar metricas.

# **1. Packages and libraries**

In [1]:
suppressMessages({
  library(dplyr)
  library(readr)
  library(sf)
  library(purrr)
  library(tibble)})

Warning messages:
1: package ‘readr’ was built under R version 4.4.3 
2: package ‘tibble’ was built under R version 4.4.3 


# **2. Load data**

SHP

In [44]:
# se trabaja en 17 S
edif_zch = st_read("../../../DATOS/SHP/INEC/BNCPV22 _edif_p_buffer_5km.shp") ## edificios
sector_acus = st_read("../../../DATOS/SHP/INEC/BNCPV22 _sec_a_ACUS.shp") ## sectores censales
grid_1km_buffer1km_acus = st_read("../../../DATOS/SHP/Grids/grid_acus_buffer_1km.shp") ## grid de 1km con buffer de 1km desde las ACUS
grid_acus_1km = st_read("../../../DATOS/SHP/Grids/grid_acus_1km.shp") ## grid de 1km con buffer de 1km desde las ACUS


Reading layer `BNCPV22 _edif_p_buffer_5km' from data source `C:\PROYECTOS\consultorias\ACUS DIAGNOSTICO\DATOS\SHP\INEC\BNCPV22 _edif_p_buffer_5km.shp' using driver `ESRI Shapefile'
Simple feature collection with 122592 features and 10 fields
Geometry type: MULTIPOINT
Dimension:     XY
Bounding box:  xmin: 669512.4 ymin: 9445399 xmax: 789693.5 ymax: 9639823
Projected CRS: WGS 84 / UTM zone 17S
Reading layer `BNCPV22 _sec_a_ACUS' from data source `C:\PROYECTOS\consultorias\ACUS DIAGNOSTICO\DATOS\SHP\INEC\BNCPV22 _sec_a_ACUS.shp' using driver `ESRI Shapefile'
Simple feature collection with 183 features and 9 fields
Geometry type: POLYGON
Dimension:     XY
Bounding box:  xmin: 674387.4 ymin: 9445216 xmax: 792744.5 ymax: 9630348
Projected CRS: WGS 84 / UTM zone 17S
Reading layer `grid_acus_buffer_1km' from data source `C:\PROYECTOS\consultorias\ACUS DIAGNOSTICO\DATOS\SHP\Grids\grid_acus_buffer_1km.shp' using driver `ESRI Shapefile'
Simple feature collection with 8535 features and 5 fields
G

CSV

In [11]:
poblacion_2022 <- readr::read_delim(
  "C:/DATOS TOTAL/INEC/2022/1.2 BDD_CPV_2022_SECTOR_CSV/CPV_2022_Poblacion_Sector.csv",
  delim = ";"
)

Rows: 16938986 Columns: 92
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ";"
chr (22): I01, I02, I04, I05, I10, INH, P00, P08P, P08C, P08Q, P09P, P09C, P09Q, P1001I, P12, P27, P28, CANTON, PARROQ, ID_VIV, ID_HOG, ID_PER
dbl (70): I03, P01, P02, P03, P05, P0601, P0602, P0701, P0702, P0703, P0704, P0705, P0706, P08, P0803A, P09, P1001, P1002, P1003, P1004, P1005, P11R...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


# **3. Filter data**

## 3.1. Filter census based on sector ID

In [ ]:
id_sector_complete = paste(poblacion_2022$I01, poblacion_2022$I02, poblacion_2022$I03, poblacion_2022$I04,poblacion_2022$I05, sep="") ## create sector id
poblacion_2022$id_sector_complete = id_sector_complete ## create variable
sector_acus_id = sector_acus$sector ## extract sector ID from ACUS

## census from acus
poblacion_acus = poblacion_2022 |>
filter(id_sector_complete %in% sector_acus_id)

In [53]:
poblacion_acus

# A tibble: 28,054 × 93
   I01   I02     I03 I04   I05   I10   INH   P00     P01   P02   P03   P05 P0601 P0602 P0701 P0702 P0703 P0704 P0705 P0706   P08 P08P  P08C  P08Q  
   <chr> <chr> <dbl> <chr> <chr> <chr> <chr> <chr> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <chr> <chr> <chr> 
 1 19    01       50 001   003   0001  01    0001      1     1    41     1     1    NA     1     1     1     1     1     1     1 19    1901  190150
 2 19    01       50 001   003   0001  01    0002      3     2    17     1     1    NA     1     1     1     1     1     1     2 18    1801  180150
 3 19    01       50 001   003   0001  01    0003      3     1    13     1     1    NA     1     1     1     1     1     1     2 09    0901  090150
 4 19    01       50 001   003   0002  01    0001      1     1    68     1     1    NA     1     1     1     2     1     1     2 11    1101  110150
 5 19    01       50 001   003   0003  01    0001      1     2    64     1     1    NA  

In [56]:
write.csv(poblacion_acus, "../../../DATOS/Datasets/INEC/poblacion_acus_censo_2022_a_sen.csv")

## 3.1. Merge Grids with grids

In [64]:

edif_acus_grid = edif_zch|> 
st_filter(grid_acus_1km, .predicate = st_within) ## mantiene edifcios que estan dentro de la cuadricula de las acus


edif_sector_acus <- edif_acus_grid |>
  st_join(sector_acus, join = st_within)|> ## extrae información de la zona censal donde esta cada edificio.
  filter(!is.na(sector)) ## elimina edificios donde no hay sectores (es decir ese sector no esta dentro de las acus)

In [67]:
st_write(
  edif_sector_acus,
  "../../../DATOS/SHP/INEC/edif_sector_acus_grids.shp",
  delete_dsn = TRUE
)

Deleting source `../../../DATOS/SHP/INEC/edif_sector_acus_grids.shp' failed
Writing layer `edif_sector_acus_grids' to data source `../../../DATOS/SHP/INEC/edif_sector_acus_grids.shp' using driver `ESRI Shapefile'
Writing 6331 features with 19 fields and geometry type Multi Point.


Warning message:
In CPL_write_ogr(obj, dsn, layer, driver, as.character(dataset_options),  :
  GDAL Error 1: ../../../DATOS/SHP/INEC/edif_sector_acus_grids.shp does not appear to be a file or directory.
